<a href="https://colab.research.google.com/github/Adhiaris/Grokking-Deep-Learning/blob/main/chapter_08_regularization_batching.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 8: Learning Signal and Ignoring Noise
### Introduction to Regularization and Batching

Networks trained too long on too little data memorize their training examples instead of generalizing. This chapter covers three techniques to prevent this: **early stopping**, **dropout**, and **batch gradient descent**.

## 1. Overfitting

A network overfits when training error keeps going down but test (unseen) error starts going up. The network has memorized the training data rather than learning the underlying pattern.

In [ ]:
import numpy as np

np.random.seed(0)

def relu(x):
    return (x > 0) * x

def relu_deriv(x):
    return x > 0

X_train = np.array([[0,1,1],[1,0,1],[1,1,1],[0,0,1],[1,0,0],[0,1,0]], dtype=float)
y_train = np.array([[1],[0],[0],[1],[0],[1]], dtype=float)
X_test  = np.array([[0,1,1],[1,1,0],[0,0,1]], dtype=float)
y_test  = np.array([[1],[0],[1]], dtype=float)

W0 = 2*np.random.rand(3, 8) - 1
W1 = 2*np.random.rand(8, 1) - 1
alpha = 0.1

print(f"{'Epoch':<8} {'Train Error':>14} {'Test Error':>12}")
print("-" * 38)

for epoch in range(300):
    l1 = relu(X_train.dot(W0))
    l2 = l1.dot(W1)
    train_err = np.sum((l2 - y_train) ** 2)

    l2_d = l2 - y_train
    l1_d = l2_d.dot(W1.T) * relu_deriv(l1)
    W1 -= alpha * l1.T.dot(l2_d)
    W0 -= alpha * X_train.T.dot(l1_d)

    l1t = relu(X_test.dot(W0))
    l2t = l1t.dot(W1)
    test_err = np.sum((l2t - y_test) ** 2)

    if epoch % 50 == 0:
        print(f"{epoch:<8} {train_err:>14.6f} {test_err:>12.6f}")

## 2. Early Stopping

The simplest regularization: stop training when test error starts increasing. This prevents the network from over-memorizing the training set.

In [ ]:
import numpy as np

np.random.seed(0)

def relu(x): return (x > 0) * x
def relu_deriv(x): return x > 0

X = np.array([[0,1,1],[1,0,1],[1,1,1],[0,0,1],[1,0,0],[0,1,0]], dtype=float)
y = np.array([[1],[0],[0],[1],[0],[1]], dtype=float)

W0 = 2*np.random.rand(3, 6) - 1
W1 = 2*np.random.rand(6, 1) - 1
alpha = 0.05

best_error = float('inf')
patience   = 0
max_patience = 20

for epoch in range(500):
    l1 = relu(X.dot(W0))
    l2 = l1.dot(W1)
    err = np.sum((l2 - y) ** 2)

    l2_d = l2 - y
    l1_d = l2_d.dot(W1.T) * relu_deriv(l1)
    W1 -= alpha * l1.T.dot(l2_d)
    W0 -= alpha * X.T.dot(l1_d)

    if err < best_error:
        best_error = err
        patience   = 0
    else:
        patience += 1

    if patience >= max_patience:
        print(f"Early stopping at epoch {epoch}")
        print(f"Best error: {best_error:.6f}")
        break

## 3. Dropout

Dropout randomly deactivates hidden neurons during each training step. This forces the network to learn redundant representations, improving generalization. At test time, all neurons are active.

In [ ]:
import numpy as np

np.random.seed(1)

def relu(x): return (x > 0) * x
def relu_deriv(x): return x > 0

streetlights = np.array([[1,0,1],[0,1,1],[0,0,1],[1,1,1],[0,1,1],[1,0,1]], dtype=float)
walk_vs_stop = np.array([[0],[1],[0],[1],[1],[0]], dtype=float)

alpha        = 0.2
dropout_prob = 0.5

W0 = 2*np.random.rand(3, 4) - 1
W1 = 2*np.random.rand(4, 1) - 1

print(f"{'Epoch':<8} {'Error':>12}")
print("-" * 22)

for j in range(80):
    l0 = streetlights
    l1 = relu(l0.dot(W0))

    dropout_mask = (np.random.rand(*l1.shape) > dropout_prob).astype(float)
    l1 *= dropout_mask * (1.0 / (1.0 - dropout_prob))

    l2 = l1.dot(W1)

    l2_err = np.sum((l2 - walk_vs_stop) ** 2)

    l2_delta = l2 - walk_vs_stop
    l1_delta = l2_delta.dot(W1.T) * relu_deriv(l1) * dropout_mask

    W1 -= alpha * l1.T.dot(l2_delta)
    W0 -= alpha * l0.T.dot(l1_delta)

    if j % 15 == 0:
        print(f"{j:<8} {l2_err:>12.6f}")

## 4. Batch Gradient Descent

Instead of updating weights after every single example (stochastic) or averaging over the full dataset (full gradient descent), batch GD updates after every N examples. This balances speed and stability.

In [ ]:
import numpy as np

np.random.seed(2)

def relu(x): return (x > 0) * x
def relu_deriv(x): return x > 0

X = np.array([[1,0,1],[0,1,1],[0,0,1],[1,1,1],[0,1,1],[1,0,1]], dtype=float)
y = np.array([[0],[1],[0],[1],[1],[0]], dtype=float)

W0 = 2*np.random.rand(3, 4) - 1
W1 = 2*np.random.rand(4, 1) - 1

alpha      = 0.1
batch_size = 2

print(f"Batch size = {batch_size}")
print(f"{'Epoch':<8} {'Error':>12}")
print("-" * 22)

for epoch in range(50):
    epoch_error = 0
    for start in range(0, len(X), batch_size):
        Xb = X[start:start+batch_size]
        yb = y[start:start+batch_size]

        l1 = relu(Xb.dot(W0))
        l2 = l1.dot(W1)

        epoch_error += np.sum((l2 - yb) ** 2)

        l2_d = l2 - yb
        l1_d = l2_d.dot(W1.T) * relu_deriv(l1)
        W1 -= alpha * l1.T.dot(l2_d)
        W0 -= alpha * Xb.T.dot(l1_d)

    if epoch % 10 == 0:
        print(f"{epoch:<8} {epoch_error:>12.6f}")

## Summary

Overfitting = memorizing training data, not learning generalizable patterns. Three tools to fight it: **early stopping** (stop when test error rises), **dropout** (randomly disable neurons during training), and **batch gradient descent** (update weights every N examples). Chapter 9 introduces activation functions to model probabilities and nonlinear relationships.